In [5]:
# Import required libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output
import dash

# Read the SpaceX data
spacex_df = pd.read_csv('data/spacex_cleaned.csv')
spacex_df['Date'] = pd.to_datetime(spacex_df['Date'])

print("Data loaded successfully!")
print(f"Dataset shape: {spacex_df.shape}")

# Get min and max payload for slider
max_payload = spacex_df['PayloadMass'].max()
min_payload = spacex_df['PayloadMass'].min()

print(f"Payload range: {min_payload:.0f} kg to {max_payload:.0f} kg")

# Create a dash application
app = Dash(__name__)

# Create app layout
app.layout = html.Div(children=[
    # Title
    html.H1(
        'SpaceX Launch Records Dashboard',
        style={
            'textAlign': 'center',
            'color': '#503D36',
            'font-size': 40,
            'font-weight': 'bold',
            'font-family': 'Arial',
            'margin-top': '20px'
        }
    ),
    
    # Subtitle
    html.Div(
        'Interactive Dashboard for Falcon 9 First Stage Landing Prediction',
        style={
            'textAlign': 'center',
            'color': '#666',
            'font-size': 18,
            'margin-bottom': '30px'
        }
    ),
    
    # Divider
    html.Hr(),
    
    # TASK 1: Dropdown for Launch Site selection
    html.Div([
        html.Label(
            'Select Launch Site:',
            style={
                'font-size': 18,
                'font-weight': 'bold',
                'margin-right': '10px'
            }
        ),
        dcc.Dropdown(
            id='site-dropdown',
            options=[
                {'label': 'All Sites', 'value': 'ALL'}
            ] + [
                {'label': site, 'value': site} 
                for site in spacex_df['LaunchSite'].unique()
            ],
            value='ALL',
            placeholder="Select a Launch Site",
            searchable=True,
            style={'width': '100%', 'padding': '5px'}
        ),
    ], style={'width': '50%', 'margin': '20px auto'}),
    
    html.Br(),
    
    # TASK 2: Pie chart for success count
    html.Div(dcc.Graph(id='success-pie-chart')),
    
    html.Br(),
    html.Hr(),
    
    # TASK 3: Payload range slider
    html.Div([
        html.Label(
            'Payload Range (kg):',
            style={
                'font-size': 18,
                'font-weight': 'bold',
                'margin-bottom': '10px'
            }
        ),
        dcc.RangeSlider(
            id='payload-slider',
            min=0,
            max=10000,
            step=500,
            marks={
                0: '0',
                2000: '2000',
                4000: '4000',
                6000: '6000',
                8000: '8000',
                10000: '10000'
            },
            value=[min_payload, max_payload],
            tooltip={"placement": "bottom", "always_visible": True}
        ),
    ], style={'width': '80%', 'margin': '20px auto'}),
    
    html.Br(),
    
    # TASK 4: Scatter plot for payload vs outcome
    html.Div(dcc.Graph(id='success-payload-scatter-chart')),
    
    html.Br(),
    html.Hr(),
    
    # Additional statistics
    html.Div(id='stats-output', style={
        'textAlign': 'center',
        'fontSize': 16,
        'margin': '20px',
        'padding': '20px',
        'backgroundColor': '#f0f0f0',
        'borderRadius': '10px'
    }),
    
    # Footer
    html.Div(
        '© 2024 SpaceX Launch Analysis Dashboard',
        style={
            'textAlign': 'center',
            'color': '#999',
            'margin-top': '50px',
            'padding': '20px'
        }
    )
])

# TASK 2: Callback for pie chart
@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value')
)
def get_pie_chart(entered_site):
    """
    Generate pie chart based on selected launch site
    """
    if entered_site == 'ALL':
        # Pie chart for all sites - show success count per site
        fig = px.pie(
            spacex_df,
            names='LaunchSite',
            values='Class',
            title='Total Success Launches by Site',
            color_discrete_sequence=px.colors.qualitative.Set3
        )
    else:
        # Pie chart for specific site - show success vs failure
        filtered_df = spacex_df[spacex_df['LaunchSite'] == entered_site]
        
        # Count successes and failures
        success_count = (filtered_df['Class'] == 1).sum()
        failure_count = (filtered_df['Class'] == 0).sum()
        
        # Create dataframe for pie chart
        pie_data = pd.DataFrame({
            'Outcome': ['Success', 'Failure'],
            'Count': [success_count, failure_count]
        })
        
        fig = px.pie(
            pie_data,
            names='Outcome',
            values='Count',
            title=f'Success vs Failure for {entered_site}',
            color='Outcome',
            color_discrete_map={'Success': '#2ecc71', 'Failure': '#e74c3c'}
        )
    
    # Update layout
    fig.update_layout(
        title_font_size=20,
        title_font_family='Arial',
        title_font_color='#503D36',
        showlegend=True,
        height=500
    )
    
    return fig

# TASK 4: Callback for scatter plot
@app.callback(
    Output(component_id='success-payload-scatter-chart', component_property='figure'),
    [Input(component_id='site-dropdown', component_property='value'),
     Input(component_id='payload-slider', component_property='value')]
)
def get_scatter_chart(entered_site, payload_range):
    """
    Generate scatter plot for payload vs outcome
    """
    low, high = payload_range
    
    # Filter by payload range
    mask = (spacex_df['PayloadMass'] >= low) & (spacex_df['PayloadMass'] <= high)
    
    if entered_site == 'ALL':
        filtered_df = spacex_df[mask]
        title_text = f'Payload vs. Launch Outcome for All Sites (Payload: {low}-{high} kg)'
    else:
        filtered_df = spacex_df[(spacex_df['LaunchSite'] == entered_site) & mask]
        title_text = f'Payload vs. Launch Outcome for {entered_site} (Payload: {low}-{high} kg)'
    
    # Create scatter plot
    fig = px.scatter(
        filtered_df,
        x='PayloadMass',
        y='Class',
        color='Rocket',
        title=title_text,
        labels={
            'PayloadMass': 'Payload Mass (kg)',
            'Class': 'Launch Outcome (0=Failure, 1=Success)',
            'Rocket': 'Booster Version'
        },
        hover_data=['Mission', 'Date', 'LaunchSite', 'Orbit_clean'],
        color_discrete_sequence=px.colors.qualitative.Bold
    )
    
    # Update layout
    fig.update_layout(
        title_font_size=18,
        title_font_family='Arial',
        title_font_color='#503D36',
        xaxis=dict(
            title_font_size=14,
            gridcolor='lightgray'
        ),
        yaxis=dict(
            title_font_size=14,
            gridcolor='lightgray',
            tickmode='array',
            tickvals=[0, 1],
            ticktext=['Failure', 'Success']
        ),
        height=600,
        hovermode='closest'
    )
    
    # Add trend line
    fig.update_traces(marker=dict(size=10, line=dict(width=1, color='DarkSlateGrey')))
    
    return fig

# Callback for statistics output
@app.callback(
    Output(component_id='stats-output', component_property='children'),
    [Input(component_id='site-dropdown', component_property='value'),
     Input(component_id='payload-slider', component_property='value')]
)
def update_stats(entered_site, payload_range):
    """
    Update statistics based on selections
    """
    low, high = payload_range
    
    # Filter data
    mask = (spacex_df['PayloadMass'] >= low) & (spacex_df['PayloadMass'] <= high)
    
    if entered_site == 'ALL':
        filtered_df = spacex_df[mask]
        site_text = "All Sites"
    else:
        filtered_df = spacex_df[(spacex_df['LaunchSite'] == entered_site) & mask]
        site_text = entered_site
    
    # Calculate statistics
    total_launches = len(filtered_df)
    successful_launches = (filtered_df['Class'] == 1).sum()
    success_rate = (successful_launches / total_launches * 100) if total_launches > 0 else 0
    avg_payload = filtered_df['PayloadMass'].mean()
    
    # Create statistics display
    stats_html = html.Div([
        html.H3('Current Selection Statistics', style={'marginBottom': '15px'}),
        html.Div([
            html.Div([
                html.H4(f'{total_launches}', style={'color': '#3498db', 'fontSize': 32, 'margin': '0'}),
                html.P('Total Launches', style={'margin': '5px 0'})
            ], style={'display': 'inline-block', 'margin': '0 30px'}),
            
            html.Div([
                html.H4(f'{successful_launches}', style={'color': '#2ecc71', 'fontSize': 32, 'margin': '0'}),
                html.P('Successful Landings', style={'margin': '5px 0'})
            ], style={'display': 'inline-block', 'margin': '0 30px'}),
            
            html.Div([
                html.H4(f'{success_rate:.1f}%', style={'color': '#e67e22', 'fontSize': 32, 'margin': '0'}),
                html.P('Success Rate', style={'margin': '5px 0'})
            ], style={'display': 'inline-block', 'margin': '0 30px'}),
            
            html.Div([
                html.H4(f'{avg_payload:.0f} kg', style={'color': '#9b59b6', 'fontSize': 32, 'margin': '0'}),
                html.P('Average Payload', style={'margin': '5px 0'})
            ], style={'display': 'inline-block', 'margin': '0 30px'}),
        ]),
        html.P(f'Filtered by: {site_text} | Payload Range: {low}-{high} kg',
               style={'marginTop': '15px', 'color': '#666', 'fontStyle': 'italic'})
    ])
    
    return stats_html

# Run the app
if __name__ == '__main__':
    print("\n" + "=" * 70)
    print("STARTING DASH APPLICATION")
    print("=" * 70)
    print("\nDashboard will be available at: http://127.0.0.1:8050/")
    print("\nPress Ctrl+C to stop the server")
    print("=" * 70 + "\n")
    
    app.run(debug=True)

Data loaded successfully!
Dataset shape: (179, 14)
Payload range: 330 kg to 15600 kg

STARTING DASH APPLICATION

Dashboard will be available at: http://127.0.0.1:8050/

Press Ctrl+C to stop the server

